# Recursion Patterns

---

## The Big WHY

we've been using recursion all week without thinking about it explicitly. Today we zoom out and recognize the patterns — because once we see them, every new tree problem becomes a variation of something you've already solved.

There are really only a few recursion shapes:

| Pattern | What it does | Examples |
|---------|-------------|----------|
| **Return a value up** | Compute something from children, combine at parent | height, maxDepth, count_nodes |
| **Pass a value down** | Parent gives context to children | isValidBST (min/max bounds) |
| **Check two trees together** | Recurse both simultaneously | isSameTree, isSubtree |
| **Find a path** | Track path from root to leaf | LCA, path sum |

Today's two LeetCode problems — **Subtree of Another Tree** and **Lowest Common Ancestor** — are the classic examples of patterns 3 and 4.

Path Sum: Does any root-to-leaf path add up to a target value?  
LCA: Find the deepest node that is an ancestor of both p and q


---

## When to use recursion vs iteration

**Use recursion when:**
- The problem has a natural tree/hierarchy structure
- Each subproblem looks identical to the original (just smaller)
- You need to combine results from children back at the parent

**Use iteration when:**
- The tree is very deep (recursion risks stack overflow)
- You need level-by-level processing (BFS)
- Python's recursion limit (1000 by default) is a concern

---

## The two-function recursion pattern

Some problems need two recursive functions — an outer one that walks the tree, and an inner one that checks something at each position. **Subtree of Another Tree** is the classic example:

```
isSubtree(root, subRoot)
  └── walks every node in root
      └── at each node, calls isSameTree(node, subRoot)
              └── checks if the two trees are identical from that point
```

---

## ML Connection — Recursive Chunking for RAG

while building a RAG system, one key step is **chunking** — splitting documents into pieces for embedding. The best chunking strategy is recursive:

```
def recursive_chunk(text, max_size, separators):
    if len(text) <= max_size:
        return [text]                          # base case — small enough
    
    separator = separators[0]                  # try splitting by paragraph
    chunks = text.split(separator)
    
    result = []
    for chunk in chunks:
        result += recursive_chunk(chunk, max_size, separators[1:])  # recurse with next separator
    return result
```

This is LangChain's `RecursiveCharacterTextSplitter` under the hood. Same recursion pattern as tree problems — just applied to text.

---

In [2]:
# helper functions: TreeNode, build_tree, tree_to_list, isSameTree
from collections import deque

class TreeNode:
    def __init__(self, val, left = None, right = None):
        self.val = val
        self.left = left
        self.right = right

    def __repr__(self):
        return f"TreeNode({self.val})"
    
def build_tree(values):
    root = TreeNode(values[0])
    queue = deque([root])
    i = 1

    while queue and i < len(values):
        node = queue.popleft()

        # left child
        if values[i] is not None:
            node.left = TreeNode(values[i])
            queue.append(node.left)
        i += 1     # always increment i, even if we we skip to match the index

        # right child
        if i < len(values):
            if values[i] is not None:
                node.right = TreeNode(values[i])
                queue.append(node.right)
            i += 1
    
    return root

def tree_to_list(root):
    queue = deque([root])
    result = []

    while queue:
        node = queue.popleft()

        if node is not None:
            result.append(node.val)
            queue.append(node.left)
            queue.append(node.right)
        else:
            result.append(None)     # represent the missing nodes, part of the structure

    ## remove the trailing none because they are just empty padding
    while result and result[-1] is None:
        result.pop()
    
    return result

def isSameTree(p, q):
    if p is None and q is None:
        return True
    if p is None or q is None:
        return False
    if p.val != q.val:
        return False
    left_check = isSameTree(p.left, q.left)
    right_check = isSameTree(p.right, q.right)

    return left_check and right_check


## LeetCode

### Problem 1: Subtree of Another Tree (LC #572)

**What it asks:** Given trees `root` and `subRoot`, return `True` if `subRoot` is a subtree of `root` — meaning some node in `root` has a subtree identical to `subRoot`.

**The two-function approach:**
- Outer function `isSubtree(root, subRoot)` — walks every node in `root`
- Inner function `isSameTree(p, q)` — checks if two trees are identical

**Logic for `isSubtree`:**
- If `root` is `None` → `False` (ran out of tree, never found it)
- If `isSameTree(root, subRoot)` → `True` (found it here)
- Otherwise → check left subtree OR right subtree

```
root:          subRoot:
    3              4
   / \            / \
  4   5          1   2
 / \
1   2

Output: True  (the subtree rooted at 4 matches subRoot)
```

In [3]:

# isSubtree(root, subRoot) → bool
# Base case: root is None → False
# Check isSameTree at current node, then recurse left and right

def isSubtree(root, subRoot):
    if root is None:
        return False
    def isSameTree(root, subRoot):
        if root is None and subRoot is None:
            return True
        if root is None or subRoot is None:
            return False
        if root.val != subRoot.val:
            return False
        left_check = isSameTree(root.left, subRoot.left)
        right_check = isSameTree(root.right, subRoot.right)

        return left_check and right_check
    if isSameTree(root, subRoot):
        return True
    left_check = isSubtree(root.left, subRoot)
    right_check = isSubtree(root.right, subRoot)
    return left_check or right_check

    # or we can use the single line of code instead of last 3 lines
    # return isSameTree(root.left, subRoot) or isSameTree(root.right, subRoot)



# Tests:
root = build_tree([3, 4, 5, 1, 2])
subRoot = build_tree([4, 1, 2])
print(isSubtree(root, subRoot))   # True

root2 = build_tree([3, 4, 5, 1, 2, None, None, None, None, 0])
print(isSubtree(root2, subRoot))  # False


True
False


### Problem 2: Lowest Common Ancestor of BST (LC #235)

**What it asks:** Given a BST and two nodes `p` and `q`, find their lowest common ancestor (LCA) — the deepest node that has both `p` and `q` as descendants.

**The BST property makes this elegant.** At each node, only three cases are possible:

- Both `p` and `q` are **less than** current node → LCA is in the left subtree
- Both `p` and `q` are **greater than** current node → LCA is in the right subtree  
- They're on **different sides** (or one equals current node) → current node IS the LCA

```
        6
       / \
      2   8
     / \ / \
    0  4 7  9
      / \
     3   5

LCA(2, 8) → 6  (they're on different sides)
LCA(2, 4) → 2  (4 is in 2's subtree, so 2 is the LCA)
```

In [5]:
# lowestCommonAncestor(root, p, q) → TreeNode
# p and q are TreeNode objects, compare using p.val and q.val
# Use the 3 cases above — no base case needed if p and q are guaranteed in the tree

def LowestCommonAncestor(root, p, q):
    if p.val < root.val and q.val < root.val:
        return LowestCommonAncestor(root.left, p, q)
    if p.val > root.val and q.val > root.val:
        return LowestCommonAncestor(root.right, p, q)
    # third case where the current node is the LCA
    return root
    

# Tests:
root = build_tree([6, 2, 8, 0, 4, 7, 9, None, None, 3, 5])
p = root.left        # node 2
q = root.right       # node 8
print(LowestCommonAncestor(root, p, q))  # TreeNode(6)

p = root.left        # node 2
q = root.left.right  # node 4
print(LowestCommonAncestor(root, p, q))  # TreeNode(2)


TreeNode(6)
TreeNode(2)


---

## Summary

| Concept | Key idea |
|---------|----------|
| Two-function pattern | Outer walks the tree, inner checks at each node |
| isSubtree | isSameTree at every node — OR the result from left/right |
| LCA in BST | BST property tells you which direction to go — no need to check both sides |
| Recursive chunking | Same pattern as tree recursion — applied to text splitting for RAG |
